# Experiment B — CNN Baseline (ConvNeXt-Tiny) on CIFAKE
### Real vs. AI-Generated Image Detection using a pretrained CNN backbone (no handcrafted features)

| Field | Value |
|---|---|
| **Experiment ID** | EXP-B-CNN-CONVNEXT-v1.0 |
| **Objective** | Learn purely from pixels via a CNN backbone — isolates what a learned representation adds over Experiment A's handcrafted features |
| **Dataset** | CIFAKE (Kaggle, Version 3) — same split as Experiment A |
| **Backbone** | `convnext_tiny` (timm, ImageNet-pretrained). Set `CONFIG["backbone"] = "efficientnet_b0"` if compute-limited |
| **Environment** | Kaggle Notebook, **T4 GPU required** |
| **Random Seed** | 42 (identical to Experiment A) |
| **Protocol** | Experiment Protocol v1.0 |

**What's different from Experiment A, and why:**
- **No handcrafted features.** The model learns directly from raw pixels — this isolates the backbone's contribution as its own variable (per the controlled-experiment design: A = handcrafted, B = CNN, C = Transformer, D = fusion).
- **Augmentation is used here** (unlike A): `RandomHorizontalFlip`, `RandomCrop(pad=4)`, light `ColorJitter`, small `RandomRotation`, optional `RandomErasing`. This is correct for deep learning — augmentation regularizes a network that would otherwise overfit CIFAKE's 32×32 images; it wasn't used in A because it would have corrupted the handcrafted descriptors' intrinsic statistics.
- **Same train/val/test split** (same seed, same stratification) as Experiment A, so results are directly comparable.
- **Same evaluation utilities** (Section 8 below) are copied verbatim from Experiment A — any metric difference is attributable to the model, not the pipeline.
- **Explainability swaps SHAP → Grad-CAM/Grad-CAM++** (appropriate for CNNs), and adds embedding visualization (t-SNE/UMAP) to inspect whether real/fake images separate in learned feature space and whether generators cluster.

This notebook is self-contained: run top to bottom on Kaggle with the CIFAKE dataset attached and **GPU accelerator enabled** (Settings → Accelerator → GPU T4 x2 or T4 x1).


## 0. Environment, Packages & Warning Suppression

In [ ]:
import subprocess, sys

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(["timm", "grad-cam", "umap-learn"])

import warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*valid feature names.*")
warnings.filterwarnings("ignore", category=FutureWarning)

import platform, torch, torchvision, timm
print("Python     :", platform.python_version())
print("PyTorch    :", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm       :", timm.__version__)
print("CUDA avail :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device     :", torch.cuda.get_device_name(0))


## 1. Configuration Block
Mirrors Experiment A's config philosophy: every tunable lives here. `subsample_per_class_*` is the same dry-run toggle you used in Experiment A — validate the full pipeline on a small slice before committing GPU time to the full run.

In [ ]:
import os, json, random
import numpy as np

CONFIG = {
    "experiment_id": "EXP-B-CNN-CONVNEXT-v1.0",
    "seed": 42,                     # identical to Experiment A
    "val_split": 0.10,              # identical to Experiment A

    # ---- Dataset paths (same as Experiment A — edit if Kaggle mounts elsewhere) ----
    "data_root": "/kaggle/input/cifake-real-and-ai-generated-synthetic-images",
    "train_dir": "train",
    "test_dir": "test",
    "classes": ["REAL", "FAKE"],    # REAL -> 0, FAKE -> 1

    # ---- Debug / dev toggle ----
    # Set to an int (e.g. 2000) to subsample for a fast dry run; None = full dataset.
    # This is exactly the toggle you used in Experiment A to validate the script end-to-end
    # before committing to a full run.
    "subsample_per_class_train": 2000,
    "subsample_per_class_test": 1000,

    # ---- Model ----
    "backbone": "convnext_tiny",    # or "efficientnet_b0" if compute-limited
    "pretrained": True,
    "img_size": 128,                # upsampled from native 32x32 for the pretrained backbone;
                                     # raise to 224 for the full run if time budget allows

    # ---- Augmentation (deep learning only — NOT used in Experiment A) ----
    "hflip_p": 0.5,
    "crop_pad": 4,                  # applied at native 32x32 before resize, CIFAR-style
    "color_jitter": 0.1,
    "rotation_deg": 10,
    "random_erasing_p": 0.1,

    # ---- Training ----
    "batch_size": 128,
    "epochs": 6,                    # bump for the full run (e.g. 15-20)
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "early_stop_patience": 3,
    "num_workers": 2,
    "use_amp": True,

    # ---- Output ----
    "output_dir": "/kaggle/working/experiment_B",
}

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_all_seeds(CONFIG["seed"])

for sub in ["model", "metrics", "figures", "logs", "gradcam", "errors", "embeddings", "report"]:
    os.makedirs(os.path.join(CONFIG["output_dir"], sub), exist_ok=True)

with open(os.path.join(CONFIG["output_dir"], "logs", "experiment_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Config saved. Output root:", CONFIG["output_dir"])


## 2. Dataset Discovery, Integrity Check & Split
Same protocol as Experiment A — identical stratified 10% validation split, same seed. If you want the *exact same* split object as Experiment A (not just the same procedure), you can instead load `experiment_A/logs/split_summary.json`-referenced paths; the procedure below reproduces it deterministically given the same file listing.

In [ ]:
import glob, pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

def list_dataset(root, split_dir, classes):
    rows = []
    for cls in classes:
        folder = os.path.join(root, split_dir, cls)
        files = glob.glob(os.path.join(folder, "*.*"))
        for fp in files:
            rows.append({"path": fp, "label_str": cls})
    return pd.DataFrame(rows)

train_df = list_dataset(CONFIG["data_root"], CONFIG["train_dir"], CONFIG["classes"])
test_df  = list_dataset(CONFIG["data_root"], CONFIG["test_dir"],  CONFIG["classes"])
train_df["label"] = (train_df["label_str"] == "FAKE").astype(int)
test_df["label"] = (test_df["label_str"] == "FAKE").astype(int)

print("Train images found:", len(train_df))
print("Test images found :", len(test_df))

if CONFIG["subsample_per_class_train"] is not None:
    train_df = train_df.groupby("label_str", group_keys=False).apply(
        lambda x: x.sample(min(len(x), CONFIG["subsample_per_class_train"]), random_state=CONFIG["seed"])
    ).reset_index(drop=True)
if CONFIG["subsample_per_class_test"] is not None:
    test_df = test_df.groupby("label_str", group_keys=False).apply(
        lambda x: x.sample(min(len(x), CONFIG["subsample_per_class_test"]), random_state=CONFIG["seed"])
    ).reset_index(drop=True)

fit_df, val_df = train_test_split(
    train_df, test_size=CONFIG["val_split"], stratify=train_df["label"], random_state=CONFIG["seed"]
)

split_summary = {
    "train_fit": len(fit_df), "train_fit_class_balance": fit_df["label_str"].value_counts().to_dict(),
    "val": len(val_df), "val_class_balance": val_df["label_str"].value_counts().to_dict(),
    "test": len(test_df), "test_class_balance": test_df["label_str"].value_counts().to_dict(),
    "seed": CONFIG["seed"],
}
print(json.dumps(split_summary, indent=2))
with open(os.path.join(CONFIG["output_dir"], "logs", "split_summary.json"), "w") as f:
    json.dump(split_summary, f, indent=2)


## 3. Sample Images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, cls in enumerate(CONFIG["classes"]):
    sample_paths = train_df[train_df["label_str"] == cls]["path"].sample(6, random_state=CONFIG["seed"]).tolist()
    for j, fp in enumerate(sample_paths):
        img = Image.open(fp)
        axes[i, j].imshow(img)
        axes[i, j].axis("off")
    axes[i, 0].set_title(cls, loc="left")
plt.suptitle("CIFAKE Samples — Row 0: REAL, Row 1: FAKE")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "figures", "sample_images.png"), dpi=150)
plt.show()


## 4. Preprocessing & Augmentation Pipeline
Deep-learning-only augmentation, applied identically across all future deep-learning experiments (B, C):

- Random Horizontal Flip
- Random Crop with padding (CIFAR-style: pad 4px at native 32×32, then crop back to 32×32)
- Light Color Jitter
- Small Random Rotation
- Random Erasing (optional, light probability)
- Resize to `img_size` for the pretrained backbone, then ImageNet normalization

Validation/test use only deterministic resize + normalization — no randomness.


In [ ]:
import torchvision.transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.RandomHorizontalFlip(p=CONFIG["hflip_p"]),
    T.RandomCrop(32, padding=CONFIG["crop_pad"]),
    T.ColorJitter(brightness=CONFIG["color_jitter"], contrast=CONFIG["color_jitter"], saturation=CONFIG["color_jitter"]),
    T.RandomRotation(degrees=CONFIG["rotation_deg"]),
    T.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=CONFIG["random_erasing_p"]),
])

eval_transform = T.Compose([
    T.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Train transform:\n", train_transform)
print("\nEval transform:\n", eval_transform)


## 5. Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CifakeDataset(Dataset):
    def __init__(self, df, transform):
        self.paths = df["path"].tolist()
        self.labels = df["label"].values.astype(np.float32)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx]

fit_ds = CifakeDataset(fit_df, train_transform)
val_ds = CifakeDataset(val_df, eval_transform)
test_ds = CifakeDataset(test_df, eval_transform)

fit_loader = DataLoader(fit_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                         num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False,
                         num_workers=CONFIG["num_workers"], pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False,
                          num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"fit: {len(fit_ds)} | val: {len(val_ds)} | test: {len(test_ds)}")
xb, yb = next(iter(fit_loader))
print("Batch shape:", xb.shape, "Label dtype:", yb.dtype)


## 6. Model — Pretrained CNN Backbone (frozen stem, fine-tuned head + late layers)

In [ ]:
import torch.nn as nn

class Detector(nn.Module):
    """Wraps a timm backbone (num_classes=0 -> pooled feature embedding) + linear head."""
    def __init__(self, backbone_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.feat_dim = self.backbone.num_features
        self.classifier = nn.Linear(self.feat_dim, 1)

    def forward_features(self, x):
        return self.backbone(x)  # (B, feat_dim) pooled embedding

    def forward(self, x):
        feats = self.forward_features(x)
        return self.classifier(feats).squeeze(1)  # logits

model = Detector(CONFIG["backbone"], pretrained=CONFIG["pretrained"]).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone: {CONFIG['backbone']} | feat_dim={model.feat_dim} | params={n_params:,} (trainable={n_trainable:,})")


## 7. Training Loop (AMP, early stopping on validation F1)

In [ ]:
import time
from sklearn.metrics import f1_score

def evaluate_loader_quick(model, loader):
    """Fast pass for validation-during-training: returns loss, probs, labels."""
    model.eval()
    all_probs, all_labels, losses = [], [], []
    criterion = nn.BCEWithLogitsLoss()
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            with torch.autocast(device_type="cuda", enabled=CONFIG["use_amp"] and device.type == "cuda"):
                logits = model(xb)
                loss = criterion(logits, yb)
            losses.append(loss.item())
            all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
            all_labels.append(yb.cpu().numpy())
    return float(np.mean(losses)), np.concatenate(all_probs), np.concatenate(all_labels)

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler(enabled=CONFIG["use_amp"] and device.type == "cuda")

history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_val_f1 = -1
best_state = None
patience_counter = 0

for epoch in range(CONFIG["epochs"]):
    model.train()
    t0 = time.time()
    running_loss = 0.0
    for xb, yb in fit_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=CONFIG["use_amp"] and device.type == "cuda"):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * xb.size(0)
    scheduler.step()

    train_loss = running_loss / len(fit_ds)
    val_loss, val_probs, val_labels = evaluate_loader_quick(model, val_loader)
    val_f1 = f1_score(val_labels, (val_probs >= 0.5).astype(int))

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1}/{CONFIG['epochs']} | train_loss={train_loss:.4f} | "
          f"val_loss={val_loss:.4f} | val_f1={val_f1:.4f} | {time.time()-t0:.1f}s")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stop_patience"]:
            print(f"Early stopping at epoch {epoch+1} (best val_f1={best_val_f1:.4f})")
            break

model.load_state_dict(best_state)
model.to(device)

log_dir = os.path.join(CONFIG["output_dir"], "logs")
with open(os.path.join(log_dir, "training_log.json"), "w") as f:
    json.dump({"history": history, "best_val_f1": best_val_f1, "backbone": CONFIG["backbone"]}, f, indent=2)

torch.save(model.state_dict(), os.path.join(CONFIG["output_dir"], "model", f"{CONFIG['backbone']}_best.pt"))

plt.figure(figsize=(6,4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("epoch"); plt.legend(); plt.title("Training curves")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["output_dir"], "figures", "training_curves.png"), dpi=150)
plt.show()


## 8. Evaluation Utilities
**Copied verbatim from Experiment A** — identical metric definitions, identical threshold-tuning procedure, identical plots. Any difference in results vs. Experiment A is attributable to the model, not the evaluation code.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, brier_score_loss, roc_curve, precision_recall_curve,
    confusion_matrix
)

def expected_calibration_error(y_true, y_prob, n_bins=15):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi if i < n_bins - 1 else y_prob <= hi)
        if mask.sum() == 0:
            continue
        acc = (y_true[mask] == (y_prob[mask] >= 0.5)).mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

def tune_threshold_for_f1(y_val, p_val):
    thresholds = np.linspace(0.01, 0.99, 197)
    best_t, best_f1 = 0.5, -1
    for t in thresholds:
        f1 = f1_score(y_val, (p_val >= t).astype(int))
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

def evaluate(y_true, p_prob, threshold):
    y_pred = (p_prob >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, p_prob)),
        "pr_auc": float(average_precision_score(y_true, p_prob)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "brier": float(brier_score_loss(y_true, p_prob)),
        "ece": expected_calibration_error(np.asarray(y_true), np.asarray(p_prob)),
        "threshold": float(threshold),
    }

def plot_diagnostics(y_true, p_prob, threshold, tag, out_dir):
    y_pred = (p_prob >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 4))
    plt.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        plt.text(j, i, str(v), ha="center", va="center")
    plt.xticks([0, 1], ["REAL", "FAKE"]); plt.yticks([0, 1], ["REAL", "FAKE"])
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Confusion Matrix — {tag}")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"confusion_matrix_{tag}.png"), dpi=150); plt.close()

    fpr, tpr, _ = roc_curve(y_true, p_prob)
    plt.figure(figsize=(5, 4)); plt.plot(fpr, tpr); plt.plot([0,1],[0,1],"--",color="gray")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"ROC Curve — {tag}")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"roc_curve_{tag}.png"), dpi=150); plt.close()

    prec, rec, _ = precision_recall_curve(y_true, p_prob)
    plt.figure(figsize=(5, 4)); plt.plot(rec, prec)
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"PR Curve — {tag}")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"pr_curve_{tag}.png"), dpi=150); plt.close()

    bins = np.linspace(0, 1, 11)
    bin_ids = np.digitize(p_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, 9)
    acc_per_bin, conf_per_bin = [], []
    for b in range(10):
        mask = bin_ids == b
        if mask.sum() == 0:
            acc_per_bin.append(0); conf_per_bin.append((bins[b]+bins[b+1])/2); continue
        acc_per_bin.append((y_true[mask] == (p_prob[mask] >= 0.5)).mean())
        conf_per_bin.append(p_prob[mask].mean())
    plt.figure(figsize=(5, 4))
    plt.plot([0,1],[0,1],"--",color="gray")
    plt.plot(conf_per_bin, acc_per_bin, marker="o")
    plt.xlabel("Mean predicted confidence"); plt.ylabel("Empirical accuracy")
    plt.title(f"Reliability Diagram — {tag}")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"calibration_curve_{tag}.png"), dpi=150); plt.close()

    plt.figure(figsize=(5, 4))
    plt.hist(p_prob, bins=20)
    plt.xlabel("Predicted P(FAKE)"); plt.ylabel("count"); plt.title(f"Confidence Histogram — {tag}")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"confidence_histogram_{tag}.png"), dpi=150); plt.close()


## 9. Threshold Tuning (Validation) & Test Evaluation

In [ ]:
def evaluate_loader_full(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            with torch.autocast(device_type="cuda", enabled=CONFIG["use_amp"] and device.type == "cuda"):
                logits = model(xb)
            all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
            all_labels.append(yb.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

p_val, y_val_arr = evaluate_loader_full(model, val_loader)
thr, val_f1 = tune_threshold_for_f1(y_val_arr, p_val)
print(f"Tuned threshold (validation only): {thr:.3f} | val F1 at threshold: {val_f1:.4f}")

p_test, y_test_arr = evaluate_loader_full(model, test_loader)

tag = f"cnn_{CONFIG['backbone']}"
metrics = evaluate(y_test_arr, p_test, thr)
metrics.update({"model": CONFIG["backbone"], "feature_set": "raw_pixels", "val_f1_at_threshold": val_f1})

metrics_dir = os.path.join(CONFIG["output_dir"], "metrics")
fig_dir = os.path.join(CONFIG["output_dir"], "figures")
plot_diagnostics(y_test_arr, p_test, thr, tag, fig_dir)
with open(os.path.join(metrics_dir, f"metrics_{tag}.json"), "w") as f:
    json.dump(metrics, f, indent=2)

results_df = pd.DataFrame([metrics])
results_df.to_csv(os.path.join(metrics_dir, "all_results.csv"), index=False)
print(json.dumps(metrics, indent=2))


## 10. Explainability — Grad-CAM & Grad-CAM++
CNN-appropriate explainability (replacing SHAP from Experiment A): visualize which spatial regions the model relies on for its REAL/FAKE decision.

In [ ]:
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

gradcam_dir = os.path.join(CONFIG["output_dir"], "gradcam")

# Generic target layer: last Conv2d anywhere in the backbone (works for ConvNeXt, EfficientNet, etc.)
target_layer = None
for name, module in model.backbone.named_modules():
    if isinstance(module, nn.Conv2d):
        target_layer = module
assert target_layer is not None, "No Conv2d layer found in backbone"

class BinaryModelWrapper(nn.Module):
    """pytorch-grad-cam expects multi-class-style outputs; wrap our single logit as 2-class."""
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
    def forward(self, x):
        logit = self.base_model(x)
        return torch.stack([-logit, logit], dim=1)  # [P(real)-like, P(fake)-like]

wrapped_model = BinaryModelWrapper(model).to(device)
cam_methods = {"gradcam": GradCAM, "gradcam_plusplus": GradCAMPlusPlus}

# pick a few correctly-classified FAKE and REAL examples for illustration
sample_idx = np.random.RandomState(CONFIG["seed"]).choice(len(test_ds), size=6, replace=False)

for cam_name, cam_cls in cam_methods.items():
    cam = cam_cls(model=wrapped_model, target_layers=[target_layer])
    fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
    for j, idx in enumerate(sample_idx):
        img_tensor, label = test_ds[idx]
        input_tensor = img_tensor.unsqueeze(0).to(device)
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]

        # unnormalize for visualization
        mean = np.array(IMAGENET_MEAN); std = np.array(IMAGENET_STD)
        img_np = img_tensor.permute(1,2,0).numpy() * std + mean
        img_np = np.clip(img_np, 0, 1).astype(np.float32)

        vis = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
        axes[j].imshow(vis); axes[j].axis("off")
        axes[j].set_title("FAKE" if label == 1 else "REAL", fontsize=9)
    plt.suptitle(f"{cam_name} — {CONFIG['backbone']}")
    plt.tight_layout()
    plt.savefig(os.path.join(gradcam_dir, f"{cam_name}_examples.png"), dpi=150)
    plt.show()
    del cam


## 11. Feature/Embedding Analysis — t-SNE & UMAP
Extract penultimate-layer embeddings and inspect: Are real/fake well separated? Does the model cluster by content instead of authenticity?

In [ ]:
from sklearn.manifold import TSNE
import umap

def extract_embeddings(model, loader):
    model.eval()
    feats, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            with torch.autocast(device_type="cuda", enabled=CONFIG["use_amp"] and device.type == "cuda"):
                f = model.forward_features(xb)
            feats.append(f.float().cpu().numpy())
            labels.append(yb.numpy())
    return np.concatenate(feats), np.concatenate(labels)

emb_test, y_emb = extract_embeddings(model, test_loader)
emb_dir = os.path.join(CONFIG["output_dir"], "embeddings")
np.savez_compressed(os.path.join(emb_dir, "test_embeddings.npz"), embeddings=emb_test, labels=y_emb)

# subsample for speed if large
max_pts = 3000
if len(emb_test) > max_pts:
    idx = np.random.RandomState(CONFIG["seed"]).choice(len(emb_test), max_pts, replace=False)
    emb_sub, y_sub = emb_test[idx], y_emb[idx]
else:
    emb_sub, y_sub = emb_test, y_emb

tsne = TSNE(n_components=2, random_state=CONFIG["seed"], init="pca", perplexity=30)
Z_tsne = tsne.fit_transform(emb_sub)

reducer = umap.UMAP(n_components=2, random_state=CONFIG["seed"])
Z_umap = reducer.fit_transform(emb_sub)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, Z, title in [(axes[0], Z_tsne, "t-SNE"), (axes[1], Z_umap, "UMAP")]:
    for lab, color, lbl in [(0, "tab:blue", "REAL"), (1, "tab:red", "FAKE")]:
        mask = y_sub == lab
        ax.scatter(Z[mask, 0], Z[mask, 1], s=6, alpha=0.5, c=color, label=lbl)
    ax.set_title(f"{title} — penultimate embeddings ({CONFIG[\'backbone\']})")
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(emb_dir, "tsne_umap_embeddings.png"), dpi=150)
plt.show()


## 12. Error Analysis
Same structure as Experiment A: false positives, false negatives, highest/lowest-confidence predictions, image galleries.

In [ ]:
err_dir = os.path.join(CONFIG["output_dir"], "errors")

y_pred_test = (p_test >= thr).astype(int)
test_df_reset = test_df.reset_index(drop=True)

err_frame = pd.DataFrame({
    "path": test_df_reset["path"],
    "y_true": y_test_arr,
    "y_pred": y_pred_test,
    "p_fake": p_test,
})
err_frame["error_type"] = np.select(
    [ (err_frame.y_true == 0) & (err_frame.y_pred == 1),
      (err_frame.y_true == 1) & (err_frame.y_pred == 0) ],
    ["false_positive", "false_negative"], default="correct"
)
err_frame["confidence"] = np.where(err_frame.y_pred == 1, err_frame.p_fake, 1 - err_frame.p_fake)
err_frame.to_csv(os.path.join(err_dir, "misclassified_samples.csv"), index=False)

fp = err_frame[err_frame.error_type == "false_positive"].sort_values("confidence", ascending=False)
fn = err_frame[err_frame.error_type == "false_negative"].sort_values("confidence", ascending=False)
fp.to_csv(os.path.join(err_dir, "false_positives.csv"), index=False)
fn.to_csv(os.path.join(err_dir, "false_negatives.csv"), index=False)

def gallery(df_subset, title, fname, n=25):
    n = min(n, len(df_subset))
    if n == 0:
        print(f"No samples for {title}"); return
    cols = 5; rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.2, rows*2.4))
    axes = np.array(axes).reshape(-1)
    for i in range(rows*cols):
        ax = axes[i]; ax.axis("off")
        if i < n:
            row = df_subset.iloc[i]
            img = Image.open(row["path"])
            ax.imshow(img)
            ax.set_title(f"p={row['p_fake']:.2f}", fontsize=8)
    plt.suptitle(title); plt.tight_layout()
    plt.savefig(os.path.join(err_dir, fname), dpi=150); plt.close()

gallery(fp.head(25), f"Top False Positives — {tag}", "false_positive_gallery.png")
gallery(fn.head(25), f"Top False Negatives — {tag}", "false_negative_gallery.png")
gallery(err_frame.sort_values("confidence").head(25), f"Lowest-Confidence Predictions — {tag}", "low_confidence_gallery.png")

print(f"FP: {len(fp)}, FN: {len(fn)} out of {len(err_frame)} test samples")


## 13. Experiment Log & Summary Report

In [ ]:
log_path = os.path.join(CONFIG["output_dir"], "report", "experiment_b_report.md")

summary_table = results_df[["model","feature_set","accuracy","precision","recall","f1",
                             "roc_auc","pr_auc","balanced_accuracy","mcc","brier","ece","threshold"]]

with open(log_path, "w") as f:
    f.write(f"# Experiment B Report — {CONFIG['experiment_id']}\n\n")
    f.write(f"Seed: {CONFIG['seed']}  \n")
    f.write(f"Backbone: **{CONFIG['backbone']}** (pretrained={CONFIG['pretrained']})  \n\n")
    f.write("## Results\n\n")
    f.write(summary_table.to_markdown(index=False))
    f.write("\n\n## Discussion\n")
    f.write("- Compare against Experiment A's `mixed` LightGBM/XGBoost row in the master log — does the learned CNN representation beat the handcrafted-feature ceiling?\n")
    f.write("- Grad-CAM: see `gradcam/gradcam_examples.png` and `gradcam/gradcam_plusplus_examples.png` — do activations concentrate on plausible forensic regions (edges, textures) or on semantic content (which would suggest the model is learning content shortcuts rather than authenticity cues)?\n")
    f.write("- Embeddings: see `embeddings/tsne_umap_embeddings.png` — are REAL/FAKE well separated? Any visible sub-clusters suggesting the model is keying on image content rather than generative artifacts?\n")
    f.write("- Calibration: see `figures/calibration_curve_*.png`.\n")

print(summary_table.to_string(index=False))
print("\nFull report written to", log_path)

master_log_path = "/kaggle/working/experiment_logs.csv"
log_rows = summary_table.copy()
log_rows.insert(0, "experiment_id", CONFIG["experiment_id"])
if os.path.exists(master_log_path):
    existing = pd.read_csv(master_log_path)
    combined = pd.concat([existing, log_rows], ignore_index=True)
else:
    combined = log_rows
combined.to_csv(master_log_path, index=False)
print("Master experiment log updated:", master_log_path)


## 14. Outputs Summary

All artifacts saved under `/kaggle/working/experiment_B/`:

```
experiment_B/
├── model/            {backbone}_best.pt
├── metrics/          metrics_cnn_*.json, all_results.csv
├── figures/           confusion_matrix / roc_curve / pr_curve / calibration_curve / confidence_histogram
│                      sample_images.png, training_curves.png
├── gradcam/           gradcam_examples.png, gradcam_plusplus_examples.png
├── embeddings/        test_embeddings.npz, tsne_umap_embeddings.png
├── errors/            misclassified_samples.csv, false_positives.csv, false_negatives.csv, *_gallery.png
├── logs/              experiment_config.json, split_summary.json, training_log.json
└── report/            experiment_b_report.md
```

**Validation workflow (same as Experiment A):** the notebook ships with `subsample_per_class_train=2000` / `subsample_per_class_test=1000` so you can run it end-to-end quickly first, confirm the pipeline and outputs look right, then set both to `None`, raise `img_size` to 224 and `epochs` to 15–20, and rerun for the full result.

**Next steps:** Experiment C (ViT) reuses this exact split, augmentation pipeline, evaluation utilities (Section 8), and error-analysis structure — swap `CONFIG["backbone"]` logic for a ViT model and swap Grad-CAM for attention-rollout/attention-map visualization. Experiment D (fusion) will combine Experiment A's cached handcrafted features with this notebook's saved embeddings (`embeddings/test_embeddings.npz`).
